# 1. Setup & Imports
This section imports required libraries, configures paths, and defines the experiment scope for FinSight LightGBM forecasting.

In [1]:
import sys
sys.path.append("../../../")

import json
from pathlib import Path

import joblib
import numpy as np
import optuna
import pandas as pd
import plotly.graph_objects as go
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
import lightgbm as lgb
from lightgbm import LGBMRegressor

from backend.data.data_loader import fetch_stock_data
from backend.data.preprocessor import FinSightPreprocessor

optuna.logging.set_verbosity(optuna.logging.INFO)

TICKER = "HDFCBANK.NS"
START = "2020-01-01"
END = "2026-05-01"
TEST_START = "2025-01-01"

safe_ticker = ''.join(ch if ch.isalnum() else '_' for ch in TICKER)
ARTIFACTS_DIR = Path("./")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Ticker: {TICKER}")
print(f"Date range: {START} to {END}")
print(f"Artifacts dir: {ARTIFACTS_DIR.resolve()}")

c:\Users\gaura\Desktop\FinSight\finsightvenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Ticker: HDFCBANK.NS
Date range: 2020-01-01 to 2026-05-01
Artifacts dir: C:\Users\gaura\Desktop\FinSight\notebooks\models\lightgbm


# 2. Load Data
Load OHLCV data using FinSight's shared data loader and inspect shape and schema.

In [2]:
df = fetch_stock_data(TICKER, START, END)
print("Shape:", df.shape)
display(df.head())
df.info()

Shape: (1567, 9)


Price,Open,High,Low,Close,Volume,Adj Close,daily_return,log_return,hl_spread
Date,,,,,,,,,
2020-01-02,639.500000,644.000000,639.500000,643.375000,6137166,609.389160,0.006374,0.006354,4.500000
2020-01-03,641.099976,642.500000,631.799988,634.200012,10855550,600.698853,-0.014261,-0.014363,10.700012
2020-01-06,630.000000,630.900024,618.000000,620.474976,10890186,587.698730,-0.021641,-0.021879,12.900024
2020-01-07,629.450012,635.724976,626.125000,630.299988,14724494,597.004822,0.015835,0.015711,9.599976
2020-01-08,623.474976,631.075012,620.025024,628.650024,11332110,595.442078,-0.002618,-0.002621,11.049988


<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1567 entries, 2020-01-02 to 2026-05-01
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Open          1567 non-null   float64
 1   High          1567 non-null   float64
 2   Low           1567 non-null   float64
 3   Close         1567 non-null   float64
 4   Volume        1567 non-null   int64  
 5   Adj Close     1567 non-null   float64
 6   daily_return  1567 non-null   float64
 7   log_return    1567 non-null   float64
 8   hl_spread     1567 non-null   float64
dtypes: float64(8), int64(1)
memory usage: 122.4 KB


# 3. Feature Engineering
Apply FinSight preprocessing and optionally enrich the feature frame using graph features if the JSON artifact exists.
The target variable is next-day Close (`Close.shift(-1)`).

In [3]:
preprocessor = FinSightPreprocessor(ticker=TICKER)
processed_df = preprocessor.fit_transform(df)

graph_features_path = ARTIFACTS_DIR / "graph_features.json"
if graph_features_path.exists():
    with graph_features_path.open("r", encoding="utf-8") as f:
        graph_features = json.load(f)
    for k, v in graph_features.items():
        processed_df[k] = float(v)
    print(f"Loaded graph features from {graph_features_path}")
else:
    graph_features = {}
    print(f"No graph features found at {graph_features_path}; continuing without them.")

model_df = processed_df.copy()
model_df["target"] = model_df["Close"].shift(-1)
model_df = model_df.dropna(subset=["target"]).replace([np.inf, -np.inf], np.nan).dropna()

print("Final model frame shape:", model_df.shape)
display(model_df.head())
print("Feature columns (excluding target):", len([c for c in model_df.columns if c != "target"]))

No graph features found at graph_features.json; continuing without them.
Final model frame shape: (1547, 15)


,Open,High,Low,Close,Volume,Adj Close,daily_return,log_return,hl_spread,Close_lag_1,Close_lag_5,Close_lag_10,rolling_mean_20,rolling_std_20,target
Date,,,,,,,,,,,,,,,
2020-01-29,-1.277790,-1.271400,-1.224250,-1.233929,-0.545518,-1.240528,0.617558,0.620802,-0.608115,-1.282271,-1.207101,-1.030540,-1.141682,-0.666851,-1.272539
2020-01-30,-1.223918,-1.283469,-1.244035,-1.272539,-0.617486,-1.275064,-0.504253,-0.496689,-0.470182,-1.232459,-1.191381,-1.017203,-1.153743,-0.654015,-1.271554
2020-01-31,-1.253518,-1.288019,-1.232086,-1.271554,-0.657571,-1.274184,-0.004874,0.003287,-0.759839,-1.271048,-1.192560,-1.054468,-1.162108,-0.614829,-1.403536
2020-02-03,-1.389483,-1.445708,-1.398789,-1.403536,-0.576056,-1.392247,-1.694632,-1.705219,-0.573632,-1.270064,-1.315766,-1.145670,-1.171675,-0.434574,-1.257765
2020-02-04,-1.385536,-1.303056,-1.319257,-1.257765,-0.240307,-1.261849,1.887106,1.861340,0.512595,-1.401976,-1.276466,-1.187054,-1.177795,-0.416229,-1.199259


Feature columns (excluding target): 14


# 4. Train/Test Split
Create a chronological split with the last partition as test data (date-based split).

In [4]:
train_df = model_df[model_df.index < "2025-01-01"]
test_df = model_df[model_df.index >= "2025-01-01"]

feature_cols = [c for c in model_df.columns if c != "target"]
X_train, y_train = train_df[feature_cols], train_df["target"]
X_test, y_test = test_df[feature_cols], test_df["target"]

print("Train size:", len(train_df))
print("Test size:", len(test_df))
print("Features:", len(feature_cols))

Train size: 1218
Test size: 329
Features: 14


# 5. Time Series Cross Validation
Run baseline cross-validation using `TimeSeriesSplit(n_splits=5)` and report fold RMSE.

In [5]:
tscv = TimeSeriesSplit(n_splits=5)
baseline_rmses = []

for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_train), start=1):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

    model = LGBMRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbose=-1,
        device="gpu"
    )
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)])

    preds = model.predict(X_val)
    rmse = float(np.sqrt(mean_squared_error(y_val, preds)))
    baseline_rmses.append(rmse)
    print(f"Fold {fold} RMSE: {rmse:.6f}")

print(f"Mean CV RMSE: {np.mean(baseline_rmses):.6f}")

Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 0.49433
Early stopping, best iteration is:
[117]	valid_0's l2: 0.485195
Fold 1 RMSE: 0.696560
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 0.0262137
Early stopping, best iteration is:
[129]	valid_0's l2: 0.0258399
Fold 2 RMSE: 0.160748
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 0.0107606
Early stopping, best iteration is:
[83]	valid_0's l2: 0.0107399
Fold 3 RMSE: 0.103633
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 0.00734218
Early stopping, best iteration is:
[74]	valid_0's l2: 0.00711563
Fold 4 RMSE: 0.084354
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 0.0599396
[200]	valid_0's l2: 0.0594076
Early stopping, best iteration is:
[154]	valid_0's l2: 0.0578143
Fold 5 RMSE: 0.240446
Mean CV RMSE: 0.257148


# 6. Hyperparameter Tuning (Optuna)
Tune LightGBM hyperparameters with 20 Optuna trials using time-series CV and GPU execution.

In [6]:
def objective(trial: optuna.Trial) -> float:
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "random_state": 42,
        "objective": "regression",
        "device": "gpu"
    }

    cv = TimeSeriesSplit(n_splits=3)
    rmses = []
    for tr_idx, val_idx in cv.split(X_train):
        X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

        model = LGBMRegressor(**params)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
        pred = model.predict(X_val)
        rmse = float(np.sqrt(mean_squared_error(y_val, pred)))
        rmses.append(rmse)

    print(f"Trial {trial.number} | RMSE: {float(np.mean(rmses)):.4f} | params: {trial.params}")
    return float(np.mean(rmses))

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20)

best_params = study.best_params
print("Best RMSE:", study.best_value)
print("Best params:")
for k, v in best_params.items():
    print(f"  {k}: {v}")

[I 2026-05-24 19:26:56,062] A new study created in memory with name: no-name-5c21e7ae-4a4e-4a14-afad-87c3a83f49b0


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[423]	valid_0's l2: 0.02111
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[536]	valid_0's l2: 0.0113502
Training until validation scores don't improve for 50 rounds


[I 2026-05-24 19:27:14,777] Trial 0 finished with value: 0.15266682434338705 and parameters: {'n_estimators': 860, 'learning_rate': 0.013555377746669258, 'max_depth': 4, 'subsample': 0.8076361328093224, 'colsample_bytree': 0.6600066665453697, 'min_child_weight': 2}. Best is trial 0 with value: 0.15266682434338705.


Early stopping, best iteration is:
[645]	valid_0's l2: 0.0425061
Trial 0 | RMSE: 0.1527 | params: {'n_estimators': 860, 'learning_rate': 0.013555377746669258, 'max_depth': 4, 'subsample': 0.8076361328093224, 'colsample_bytree': 0.6600066665453697, 'min_child_weight': 2}
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[19]	valid_0's l2: 0.0185776
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[114]	valid_0's l2: 0.0107714
Training until validation scores don't improve for 50 rounds


[I 2026-05-24 19:27:19,670] Trial 1 finished with value: 0.14905101378622856 and parameters: {'n_estimators': 114, 'learning_rate': 0.2725267381656758, 'max_depth': 7, 'subsample': 0.7547210621082974, 'colsample_bytree': 0.852666741869061, 'min_child_weight': 2}. Best is trial 1 with value: 0.14905101378622856.


...

Early stopping, best iteration is:
[44]	valid_0's l2: 0.040839
Trial 17 | RMSE: 0.1491 | params: {'n_estimators': 485, 'learning_rate': 0.20393183670810577, 'max_depth': 10, 'subsample': 0.999434291534695, 'colsample_bytree': 0.8047904997114795, 'min_child_weight': 9}
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[44]	valid_0's l2: 0.021486
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[219]	valid_0's l2: 0.0108622
Training until validation scores don't improve for 50 rounds


[I 2026-05-24 19:29:42,573] Trial 18 finished with value: 0.15215121661358288 and parameters: {'n_estimators': 233, 'learning_rate': 0.1114328182635522, 'max_depth': 9, 'subsample': 0.9057786616598864, 'colsample_bytree': 0.6114279112076946, 'min_child_weight': 8}. Best is trial 5 with value: 0.14664132141277136.


Early stopping, best iteration is:
[61]	valid_0's l2: 0.0422921
Trial 18 | RMSE: 0.1522 | params: {'n_estimators': 233, 'learning_rate': 0.1114328182635522, 'max_depth': 9, 'subsample': 0.9057786616598864, 'colsample_bytree': 0.6114279112076946, 'min_child_weight': 8}
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[15]	valid_0's l2: 0.0195851
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[108]	valid_0's l2: 0.0109632
Training until validation scores don't improve for 50 rounds


[I 2026-05-24 19:29:47,057] Trial 19 finished with value: 0.1491039218125887 and parameters: {'n_estimators': 763, 'learning_rate': 0.2593993148078223, 'max_depth': 5, 'subsample': 0.8459375098537338, 'colsample_bytree': 0.7604843786718236, 'min_child_weight': 10}. Best is trial 5 with value: 0.14664132141277136.


Early stopping, best iteration is:
[53]	valid_0's l2: 0.0410709
Trial 19 | RMSE: 0.1491 | params: {'n_estimators': 763, 'learning_rate': 0.2593993148078223, 'max_depth': 5, 'subsample': 0.8459375098537338, 'colsample_bytree': 0.7604843786718236, 'min_child_weight': 10}
Best RMSE: 0.14664132141277136
Best params:
  n_estimators: 234
  learning_rate: 0.2846440219133672
  max_depth: 7
  subsample: 0.8306142006686434
  colsample_bytree: 0.8116386963844514
  min_child_weight: 8


# 7. Final Model Training
Train the final model with best Optuna parameters on the full training set and evaluate on the holdout test set.

In [7]:
final_params = dict(best_params)
final_params.update({
    "random_state": 42,
    "objective": "regression",
    "device": "gpu"
})

final_model = LGBMRegressor(**final_params)
final_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)])

test_pred = final_model.predict(X_test)
rmse = float(np.sqrt(mean_squared_error(y_test, test_pred)))
mae = float(mean_absolute_error(y_test, test_pred))
mape = float(np.mean(np.abs((y_test - test_pred) / y_test.replace(0, np.nan))) * 100)

print(f"RMSE: {rmse:.6f}")
print(f"MAE:  {mae:.6f}")
print(f"MAPE: {mape:.4f}%")

print("Retraining on full dataset...")
X_full = pd.concat([X_train, X_test])
y_full = pd.concat([y_train, y_test])
final_model.fit(X_full, y_full, callbacks=[lgb.log_evaluation(100)])
print("Final model trained on full dataset")

# Auto-save
model_path = ARTIFACTS_DIR / f"{safe_ticker}_lightgbm_model.pkl"
best_params_path = ARTIFACTS_DIR / f"{safe_ticker}_lgbm_best_params.json"
feature_cols_path = ARTIFACTS_DIR / f"{safe_ticker}_feature_columns.json"

joblib.dump(final_model, model_path)
preprocessor.save_artifacts()

with feature_cols_path.open("w", encoding="utf-8") as f:
    json.dump(feature_cols, f, indent=2)

with best_params_path.open("w", encoding="utf-8") as f:
    json.dump(best_params, f, indent=2)

print(f"AUTO SAVED as {safe_ticker}_*")
print(f"- Model: {model_path.resolve()}")
print(f"- Feature columns: {feature_cols_path.resolve()}")
print(f"- Best params: {best_params_path.resolve()}")

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[19]	valid_0's l2: 0.179807
RMSE: 0.424037
MAE:  0.344838
MAPE: 34.4972%
Retraining on full dataset...
Final model trained on full dataset
AUTO SAVED as HDFCBANK_NS_*
- Model: C:\Users\gaura\Desktop\FinSight\notebooks\models\lightgbm\HDFCBANK_NS_lightgbm_model.pkl
- Feature columns: C:\Users\gaura\Desktop\FinSight\notebooks\models\lightgbm\HDFCBANK_NS_feature_columns.json
- Best params: C:\Users\gaura\Desktop\FinSight\notebooks\models\lightgbm\HDFCBANK_NS_lgbm_best_params.json


# 8. Feature Importance
Visualize the top 15 most important predictors from the trained LightGBM model.

In [8]:
importance = pd.Series(final_model.feature_importances_, index=feature_cols).sort_values(ascending=False).head(15)

fig_imp = go.Figure(
    go.Bar(
        x=importance.values[::-1],
        y=importance.index[::-1],
        orientation="h",
        marker_color="#2563eb"
    )
)
fig_imp.update_layout(
    title=f"{TICKER} LightGBM Top 15 Feature Importance",
    template="plotly_white",
    xaxis_title="Importance",
    yaxis_title="Feature",
    height=600
)
out_path = ARTIFACTS_DIR / f"{safe_ticker}_lgbm_feature_importance.html"
fig_imp.write_html(str(out_path))
print(f"Wrote feature importance to {out_path.resolve()}")

Wrote feature importance to C:\Users\gaura\Desktop\FinSight\notebooks\models\lightgbm\HDFCBANK_NS_lgbm_feature_importance.html


# 9. Forecast Plot
Plot holdout-period actual vs predicted values and a 30-day future forecast using the latest feature row as a simple forward scenario.

In [9]:
future_steps = 30
last_row = X_test.iloc[[-1]] if len(X_test) > 0 else X_train.iloc[[-1]]
future_X = pd.concat([last_row] * future_steps, ignore_index=True)
future_pred = final_model.predict(future_X)

if isinstance(test_df.index, pd.DatetimeIndex):
    future_idx = pd.bdate_range(start=test_df.index[-1] + pd.tseries.offsets.BDay(1), periods=future_steps)
else:
    future_idx = pd.RangeIndex(start=len(test_df), stop=len(test_df) + future_steps)

fig_fc = go.Figure()
fig_fc.add_trace(go.Scatter(x=test_df.index, y=y_test, mode="lines", name="Actual", line=dict(color="#111827", width=2)))
fig_fc.add_trace(go.Scatter(x=test_df.index, y=test_pred, mode="lines", name="Predicted", line=dict(color="#2563eb", width=2)))
fig_fc.add_trace(go.Scatter(x=future_idx, y=future_pred, mode="lines", name="30-Day Forecast", line=dict(color="#10b981", width=2, dash="dash")))
fig_fc.update_layout(
    title=f"{TICKER} Actual vs Predicted + 30-Day Forecast",
    template="plotly_white",
    xaxis_title="Date",
    yaxis_title="Close Price",
    height=600
)
out_path = ARTIFACTS_DIR / f"{safe_ticker}_lgbm_forecast.html"
fig_fc.write_html(str(out_path))
print(f"Wrote forecast plot to {out_path.resolve()}")

Wrote forecast plot to C:\Users\gaura\Desktop\FinSight\notebooks\models\lightgbm\HDFCBANK_NS_lgbm_forecast.html


# 10. Save Artifacts
Persist the trained model, preprocessing artifacts, feature columns, and best hyperparameters under the local artifacts directory.

In [10]:
model_path = ARTIFACTS_DIR / f"{safe_ticker}_lightgbm_model.pkl"
feature_cols_path = ARTIFACTS_DIR / f"{safe_ticker}_feature_columns.json"
best_params_path = ARTIFACTS_DIR / f"{safe_ticker}_lgbm_best_params.json"

joblib.dump(final_model, model_path)
preprocessor.save_artifacts()

with feature_cols_path.open("w", encoding="utf-8") as f:
    json.dump(feature_cols, f, indent=2)

with best_params_path.open("w", encoding="utf-8") as f:
    json.dump(best_params, f, indent=2)

print("Saved artifacts:")
print(f"- Model: {model_path.resolve()}")
print(f"- Preprocessor scaler: {(ARTIFACTS_DIR / 'scaler.pkl').resolve()}")
print(f"- Feature columns: {feature_cols_path.resolve()}")
print(f"- Best params: {best_params_path.resolve()}")

Saved artifacts:
- Model: C:\Users\gaura\Desktop\FinSight\notebooks\models\lightgbm\HDFCBANK_NS_lightgbm_model.pkl
- Preprocessor scaler: C:\Users\gaura\Desktop\FinSight\notebooks\models\lightgbm\scaler.pkl
- Feature columns: C:\Users\gaura\Desktop\FinSight\notebooks\models\lightgbm\HDFCBANK_NS_feature_columns.json
- Best params: C:\Users\gaura\Desktop\FinSight\notebooks\models\lightgbm\HDFCBANK_NS_lgbm_best_params.json
